# Classifying Penguins with Keras

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [14]:
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [17]:
# Drop Nan rows
penguins = penguins.dropna()

In [39]:
# Shuffle the data
penguins = penguins.sample(frac=1, random_state=42).reset_index(drop=True)

In [40]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,2900.0,38.6,17.0,188.0,True,False
1,2900.0,34.5,18.1,187.0,True,False
2,3950.0,51.9,19.5,206.0,False,True
3,4500.0,42.5,20.7,197.0,False,True
4,4400.0,41.3,21.1,195.0,False,True
...,...,...,...,...,...,...
328,3550.0,50.9,19.1,196.0,False,True
329,3950.0,40.5,18.9,180.0,False,True
330,5450.0,52.5,15.6,221.0,False,True
331,4500.0,53.5,19.9,205.0,False,True


In [41]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.055556,0.236364,0.464286,0.271186,1.0,0.0
1,0.055556,0.087273,0.595238,0.254237,1.0,0.0
2,0.347222,0.720000,0.761905,0.576271,0.0,1.0
3,0.500000,0.378182,0.904762,0.423729,0.0,1.0
4,0.472222,0.334545,0.952381,0.389831,0.0,1.0
...,...,...,...,...,...,...
328,0.236111,0.683636,0.714286,0.406780,0.0,1.0
329,0.347222,0.305455,0.690476,0.135593,0.0,1.0
330,0.763889,0.741818,0.297619,0.830508,0.0,1.0
331,0.500000,0.778182,0.809524,0.559322,0.0,1.0


In [42]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1         Adelie
2      Chinstrap
3         Adelie
4         Adelie
         ...    
328    Chinstrap
329       Adelie
330       Gentoo
331    Chinstrap
332    Chinstrap
Name: species, Length: 333, dtype: str


array([0, 0, 1, 0, 0, 2, 0, 1, 2, 0, 2, 2, 2, 0, 2, 2, 2, 2, 0, 2, 2, 0,
       2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 0, 1, 2, 2, 2, 0, 1, 0, 0, 2, 1, 0,
       2, 2, 1, 0, 1, 0, 0, 0, 1, 0, 0, 2, 2, 2, 1, 0, 1, 1, 2, 2, 0, 2,
       1, 1, 0, 1, 2, 0, 2, 0, 1, 0, 2, 1, 1, 1, 0, 0, 0, 2, 2, 0, 1, 2,
       2, 2, 0, 0, 1, 2, 2, 0, 1, 0, 0, 2, 1, 0, 0, 0, 1, 0, 2, 0, 0, 0,
       0, 2, 2, 2, 0, 2, 0, 2, 0, 0, 0, 0, 2, 0, 1, 2, 0, 2, 1, 0, 0, 2,
       2, 0, 2, 2, 0, 2, 1, 2, 1, 1, 0, 0, 0, 0, 2, 2, 2, 0, 0, 1, 2, 0,
       0, 0, 2, 0, 1, 2, 2, 1, 2, 0, 2, 2, 2, 2, 2, 0, 2, 0, 1, 0, 2, 0,
       0, 2, 1, 0, 0, 2, 1, 2, 0, 0, 0, 0, 2, 1, 2, 2, 2, 0, 2, 1, 1, 2,
       0, 0, 2, 0, 2, 2, 0, 0, 0, 0, 0, 0, 2, 0, 2, 2, 2, 1, 2, 0, 0, 1,
       2, 2, 0, 2, 0, 0, 0, 0, 2, 0, 2, 2, 1, 2, 0, 0, 1, 0, 0, 2, 0, 1,
       0, 0, 2, 0, 1, 0, 2, 1, 0, 0, 2, 1, 1, 1, 0, 0, 2, 0, 1, 2, 0, 2,
       2, 0, 0, 1, 2, 2, 2, 1, 2, 1, 2, 0, 1, 2, 2, 1, 0, 0, 0, 1, 1, 0,
       2, 1, 1, 1, 2, 1, 0, 0, 2, 1, 2, 0, 0, 0, 2,

In [43]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [44]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [45]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [46]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


c:\Users\ldcal\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.3645 - loss: 1.0980 - val_accuracy: 0.7059 - val_loss: 1.0910
Epoch 2/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5156 - loss: 1.0936

c:\Users\ldcal\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5619 - loss: 1.0916 - val_accuracy: 0.7353 - val_loss: 1.0862
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6020 - loss: 1.0879 - val_accuracy: 0.7059 - val_loss: 1.0818
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6087 - loss: 1.0839 - val_accuracy: 0.7059 - val_loss: 1.0771
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6054 - loss: 1.0795 - val_accuracy: 0.7647 - val_loss: 1.0709
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6856 - loss: 1.0732 - val_accuracy: 0.6471 - val_loss: 1.0652
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6622 - loss: 1.0664 - val_accuracy: 0.5882 - val_loss: 1.0591
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6154 - loss: 1.0598 - val_accuracy: 0.5882 - val_loss: 1.0531
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6154 - loss: 1.0530 - val_accuracy: 0.5588 - val_loss: 1.0473
Epoc

In [55]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 5, validation_split = 0.1)

scores = model_logit_true.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.9967 - loss: 0.0127 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9967 - loss: 0.0119 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9933 - loss: 0.0114 - val_accuracy: 1.0000 - val_loss: 0.0023
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9967 - loss: 0.0118 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9967 - loss: 0.0112 - val_accuracy: 1.0000 - val_loss: 0.0031
11/11 - 0s - 7ms/step - accuracy: 0.9970 - loss: 0.0101


In [50]:
model_logit_false = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_false.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_false = model_logit_false.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_false.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.9866 - loss: 0.1061 - val_accuracy: 1.0000 - val_loss: 0.0803
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9866 - loss: 0.1038 - val_accuracy: 1.0000 - val_loss: 0.0794
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9833 - loss: 0.1003 - val_accuracy: 1.0000 - val_loss: 0.0807
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9866 - loss: 0.0986 - val_accuracy: 1.0000 - val_loss: 0.0771
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9866 - loss: 0.0971 - val_accuracy: 1.0000 - val_loss: 0.0756
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9866 - loss: 0.0956 - val_accuracy: 1.0000 - val_loss: 0.0727
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9833 - loss: 0.0946 - val_accuracy: 1.0000 - val_loss: 0.0746
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9833 - loss: 0.0927 - val_accuracy: 1.0000 - val_loss:

In [11]:
model_logit_true.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[nan, nan, nan],
       [nan, nan, nan],
       [nan, nan, nan],
       ...,
       [nan, nan, nan],
       [nan, nan, nan],
       [nan, nan, nan]], dtype=float32)

In [49]:
penguins['species']

0         Adelie
1         Adelie
2      Chinstrap
3         Adelie
4         Adelie
         ...    
328    Chinstrap
329       Adelie
330       Gentoo
331    Chinstrap
332    Chinstrap
Name: species, Length: 333, dtype: str

In [48]:
penguins_y

array([0, 0, 1, 0, 0, 2, 0, 1, 2, 0, 2, 2, 2, 0, 2, 2, 2, 2, 0, 2, 2, 0,
       2, 0, 1, 0, 2, 0, 1, 0, 2, 0, 0, 1, 2, 2, 2, 0, 1, 0, 0, 2, 1, 0,
       2, 2, 1, 0, 1, 0, 0, 0, 1, 0, 0, 2, 2, 2, 1, 0, 1, 1, 2, 2, 0, 2,
       1, 1, 0, 1, 2, 0, 2, 0, 1, 0, 2, 1, 1, 1, 0, 0, 0, 2, 2, 0, 1, 2,
       2, 2, 0, 0, 1, 2, 2, 0, 1, 0, 0, 2, 1, 0, 0, 0, 1, 0, 2, 0, 0, 0,
       0, 2, 2, 2, 0, 2, 0, 2, 0, 0, 0, 0, 2, 0, 1, 2, 0, 2, 1, 0, 0, 2,
       2, 0, 2, 2, 0, 2, 1, 2, 1, 1, 0, 0, 0, 0, 2, 2, 2, 0, 0, 1, 2, 0,
       0, 0, 2, 0, 1, 2, 2, 1, 2, 0, 2, 2, 2, 2, 2, 0, 2, 0, 1, 0, 2, 0,
       0, 2, 1, 0, 0, 2, 1, 2, 0, 0, 0, 0, 2, 1, 2, 2, 2, 0, 2, 1, 1, 2,
       0, 0, 2, 0, 2, 2, 0, 0, 0, 0, 0, 0, 2, 0, 2, 2, 2, 1, 2, 0, 0, 1,
       2, 2, 0, 2, 0, 0, 0, 0, 2, 0, 2, 2, 1, 2, 0, 0, 1, 0, 0, 2, 0, 1,
       0, 0, 2, 0, 1, 0, 2, 1, 0, 0, 2, 1, 1, 1, 0, 0, 2, 0, 1, 2, 0, 2,
       2, 0, 0, 1, 2, 2, 2, 1, 2, 1, 2, 0, 1, 2, 2, 1, 0, 0, 0, 1, 1, 0,
       2, 1, 1, 1, 2, 1, 0, 0, 2, 1, 2, 0, 0, 0, 2,